# Ground state of the 2D Fermi-Hubbard model benchmark

This notebook benchmarks quantum phase estimation (QPE) for the ground-state of the
**Fermi-Hubbard model on a two-dimensional square lattice**, and estimates the fault-tolerant resources required to run it for lattices from $2\times2$ up to $200\times200$ sites.

$$
H = -T\sum_{\langle i,j\rangle,\;\sigma\in\{\uparrow,\downarrow\}} \left(c^{\dagger}_{i\sigma}c_{j\sigma} + \text{h.c.}\right)
  \; + \; U\sum_{i} n_{i\uparrow}\,n_{i\downarrow},
\qquad n_{i\sigma}=c^{\dagger}_{i\sigma}c_{i\sigma}
$$

Here $c^{\dagger}_{i\sigma}$ and $c_{j\sigma}$ are the fermionic creation and annihilation operators on lattice sites $i,j$, and $n_{i\sigma}$ is the on-site occupation with spin $\sigma$.

**Benchmark specification**

| Quantity | Value |
|---|---|
| Lattice | 2D square, periodic in both directions, $N=L^{2}$ sites, $L \in \{2,4,6,8,10,20,30,\dots,200\}$ |
| Parameters | $U/T = 1/8$, $f = 0.875$, i.e. $2fN$ electrons |
| Initial state | Hartree-Fock (Fermi-sea) determinant |
| Target | Ground-state energy |
| Hardware model | Majorana qubits, measurement error rate $10^{-5}$ |

**References**

- Campbell, Earl T. "Early fault-tolerant simulations of the Hubbard model." *Quantum Science & Technology* **7**.1 (2022): 015007. [arXiv:2012.09238](https://arxiv.org/abs/2012.09238)
- Bärtschi, Andreas, et al. "Potential applications of quantum computing at Los Alamos National Laboratory." (2024), Chapter 5, Application 1. [arXiv:2406.06625](https://arxiv.org/abs/2406.06625)

**Requirements**

```bash
pip install 'qdk-chemistry[jupyter,qre]'
```


In [1]:
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef,
    LatticeGraph,
    MajoranaMapping,
)
from qdk_chemistry.utils import Logger
from qdk_chemistry.utils.model_hamiltonians import create_hubbard_hamiltonian

Logger.set_global_level(Logger.LogLevel.off)

HOPPING_T = 1.0                                          # T > 0, energies are quoted in units of T
U_OVER_T = 1.0 / 8.0                                     # on-site repulsion / hopping
COULOMB_U = U_OVER_T * HOPPING_T
FILLING = 0.875                                          # 2 * FILLING * N electrons in total
LATTICE_SIZES = list(range(2, 11, 2)) + list(range(20, 201, 10))
TARGET_PRECISION_PER_SITE = 0.0051   # energy accuracy per lattice site, in units of T

def target_precision(size: int) -> float:
    """Absolute ground-state energy accuracy required of an L x L lattice."""
    return TARGET_PRECISION_PER_SITE * size * size


print(f"Hubbard model: U/T = {U_OVER_T:g}  (T = {HOPPING_T:g}, U = {COULOMB_U:g}) Filling factor f = {FILLING:g}")
print(f"Lattice sizes L = {LATTICE_SIZES}")

Hubbard model: U/T = 0.125  (T = 1, U = 0.125) Filling factor f = 0.875
Lattice sizes L = [2, 4, 6, 8, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200]


## The lattice and the qubit Hamiltonian

`LatticeGraph.square` builds the $L\times L$ lattice and `create_hubbard_hamiltonian` turns it into a
fermionic Hamiltonian. The Jordan-Wigner mapping then produces the qubit Hamiltonian on $2N$ qubits
(one per spin-orbital).

In [2]:
def qubit_operator(size: int):
    num_sites = size * size
    lattice = LatticeGraph.square(size, size, periodic_x=True, periodic_y=True)
    hamiltonian = create_hubbard_hamiltonian(
        lattice, epsilon=0.0, t=HOPPING_T, U=COULOMB_U
    )
    operator = create("qubit_mapper").run(
        hamiltonian, mapping=MajoranaMapping.jordan_wigner(2 * num_sites)
    )
    return operator

## Trotter and QPE
$$
m = \left\lceil \log_2\frac{2\lambda}{\epsilon} \right\rceil,
$$
$$
t_0 = \frac{\pi}{\lambda},
$$

In [3]:
TROTTER_ORDER = 2                    # Suzuki-Trotter product-formula order

def qpe_circuit_builder(num_bits: int, evolution_time: float, num_divisions: int, operator, initial_state):
    builder = create(
        "qpe_circuit_builder",
        "qdk_standard",
        unitary_builder=AlgorithmRef(
            "hamiltonian_unitary_builder",
            "trotter",
            order=TROTTER_ORDER,
            time=evolution_time,
            num_divisions=num_divisions,
        ),
        controlled_circuit_mapper=AlgorithmRef("controlled_circuit_mapper", "pauli_sequence"),
        num_bits=num_bits,
    )
    circuit = builder.run(initial_state, operator)[0]
    return circuit

## Physical resource estimation

In [4]:
from qdk.qre import PSSPC, LatticeSurgery, estimate, plot_estimates
from qdk.qre.application import QSharpApplication
from qdk.qre.models import Majorana, RoundBasedFactory, ThreeAux

MAJORANA_ERROR_RATE = 1e-5
ARCHITECTURE = Majorana(error_rate=MAJORANA_ERROR_RATE)
MAX_ERROR = 0.01

def estimate_physical(circuit, name: str):
    """Fastest point on the qubit/runtime Pareto frontier for the given logical counts."""
    application = QSharpApplication(circuit)
    trace_query = (
        application.q()
        * PSSPC.q()
        * LatticeSurgery.q()
    )
    isa_query = ThreeAux.q() * RoundBasedFactory.q(
        code_query=ThreeAux.q()
    )
    table = estimate(application, ARCHITECTURE, isa_query, trace_query, max_error=MAX_ERROR, name=name)
    return table


rows = []
for size in LATTICE_SIZES:
    circuit = qpe_circuit_builder(
        num_bits=10,
        evolution_time=1.0,
        num_divisions=10,
        operator=qubit_operator(size),
        initial_state=None,  # Placeholder for the initial state
    )
    table = estimate_physical(circuit, f"{size}x{size}")
    rows.append(table)
plot_estimates(table)
       


AttributeError: 'NoneType' object has no attribute '_qsharp_op'

We could use FOQCS or SOSSA for the same problem in the future.
[2601.18767v1] Practical block encodings of matrix polynomials that can also be trivially controlled
[2602.05069v1] Near-frustration-free electronic structure Hamiltonian representations and lower bound certificates
